In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import re

# 1. Apply the project styling rule from Step 5.2
plt.rcParams.update({
    "figure.facecolor": "#F8F5EF",
    "axes.facecolor": "#F8F5EF",
    "axes.edgecolor": "#2B2B2B",
    "text.color": "#2B2B2B",
    "axes.labelcolor": "#2B2B2B",
    "xtick.color": "#2B2B2B",
    "ytick.color": "#2B2B2B",
    "grid.color": "#DCD5C9",
})

# 2. Load the dataset into memory safely (from PART 4)
cols = ["cord_uid", "title", "abstract", "publish_time",
        "authors", "journal", "license", "doi", "source_x"]

dtypes = {
    "cord_uid": "string",
    "title": "string",
    "abstract": "string",
    "authors": "string",
    "journal": "string",
    "license": "category",
    "doi": "string",
    "source_x": "category",
}

df = pd.read_csv("../data/metadata.csv", usecols=cols, dtype=dtypes, low_memory=False)
df["publish_time"] = pd.to_datetime(df["publish_time"], errors="coerce")
df = df.dropna(subset=["abstract"])

# 3. Create clean_abstract so Part 8 runs without KeyError
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_abstract"] = df["abstract"].apply(clean_text)

In [2]:
from bertopic import BERTopic

sample = df["clean_abstract"].dropna().sample(15000, random_state=42).tolist()

topic_model = BERTopic(language="english", min_topic_size=50, verbose=True)
topics, probs = topic_model.fit_transform(sample)

topic_info = topic_model.get_topic_info().head(10)
print(topic_info)

fig = topic_model.visualize_barchart(top_n_topics=8)
fig.update_layout(
    plot_bgcolor="#F8F5EF",
    paper_bgcolor="#F8F5EF",
    font_color="#2B2B2B",
)
fig.write_html("../figures/topic_barchart.html")

2026-09-23 13:38:45,992 - BERTopic - Embedding - Transforming documents to embeddings.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

2026-09-23 13:56:39,314 - BERTopic - Embedding - Completed ✓
2026-09-23 13:56:39,316 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-23 13:58:00,341 - BERTopic - Dimensionality - Completed ✓
2026-09-23 13:58:00,345 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-23 13:58:02,901 - BERTopic - Cluster - Completed ✓
2026-09-23 13:58:02,921 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-23 13:58:08,993 - BERTopic - Representation - Completed ✓


   Topic  Count                                  Name  \
0     -1   7268                      -1_the_and_of_in   
1      0    587        0_anxiety_mental_stress_health   
2      1    586            1_patients_surgery_the_was   
3      2    502  2_students_learning_education_online   
4      3    471               3_the_model_of_epidemic   
5      4    382     4_care_telehealth_telemedicine_to   
6      5    360                     5_in_of_the_virus   
7      6    323            6_learning_images_the_deep   
8      7    262                   7_masks_mask_the_of   
9      8    208         8_patients_with_mortality_icu   

                                      Representation  \
0  [the, and, of, in, to, for, with, is, covid, t...   
1  [anxiety, mental, stress, health, and, psychol...   
2  [patients, surgery, the, was, of, in, and, sur...   
3  [students, learning, education, online, teachi...   
4  [the, model, of, epidemic, cases, to, number, ...   
5  [care, telehealth, telemedicine, 

In [3]:
df[["title", "abstract", "journal", "publish_time", "license"]].to_csv(
    "../data/metadata_clean.csv", index=False
)